> Derived stage notebook, prepared 20 November 2026. Setup, data preparation, duplicate/split checks and overlap lexicons only.
This is a technical subset from the completed notebook, not historical proof of authorship. Place train.csv.gz, test.csv, business_lexicon.csv and science_tech_lexicon.csv beside this notebook; run in order. It writes outputs/ and can overwrite generated artifacts.


# Beyond Keywords
Investigating Business–Technology Confusion in News Topic Classification

Compare TF-IDF + Logistic Regression with frozen Sentence-BERT + Logistic Regression on **original English headlines only**: Business versus Science & Technology.

## Purpose and target users
This system helps readers and news content organizers sort English headlines into Business or Science & Technology. Headlines that combine financial and technical vocabulary can be difficult to categorize. We investigate how two classification methods handle these headlines.

## Experimental question
Does frozen Sentence-BERT with Logistic Regression outperform TF-IDF with Logistic Regression on headlines containing both Business and Science & Technology vocabulary?

We compare accuracy and macro F1 on the same held-out test headlines, reporting results for the full test set, the vocabulary-overlap subset, and the non-overlap subset.

## Five input files
Keep this notebook, `train.csv.gz`, `test.csv`, `business_lexicon.csv`, and `science_tech_lexicon.csv` in one folder. Open Jupyter in that folder and choose **Run All**. All project Python code and documentation are in this notebook; no `src`, `tests`, or supporting documents are required.

`train.csv.gz` contains the original training CSV compressed to save space. Python reads it directly; do not extract it. Both datasets retain the original three fields: class index, title, description. Only classes 3 and 4 and only titles enter the experiment. Development is split from training after duplicate control.

The notebook creates one `outputs/` folder for processed data, fitted models, caches, audits and current results. These are generated files, not extra inputs. `REBUILD=True` rebuilds the experiment; after a successful run, `False` reuses local outputs. Internet is needed for the pinned SBERT encoder on first use, not to download the supplied news data.

## Setup
Recommended: Python 3.12 in a separate environment. Install once in Anaconda Prompt or a terminal (not a notebook diagnostic cell):

```bash
python -m pip install "numpy>=2.1,<3" "pandas>=2.2,<3" "scipy>=1.14,<2" "scikit-learn>=1.6,<2" "joblib>=1.4,<2" "torch>=2.6,<3" "sentence-transformers>=6.3,<7" "transformers>=5,<6" "huggingface-hub>=1.3,<3" "matplotlib>=3.10,<4" jupyterlab socksio
python -m jupyter lab
```

Package versions are recorded in the Setup section. Saved outputs show the results of the completed run. To reproduce the experiment, restart the kernel and run all cells in order.

In [ ]:
#Beyond Keywords: transparent headline classification and evaluation pipeline.
from pathlib import Path
import csv, gzip, hashlib, html, json, re, time, unicodedata, urllib.request, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.exceptions import ConvergenceWarning

ROOT = Path.cwd()
LABELS = ['Business', 'Science & Technology']
SEED = 42
MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
REVISION = '1110a243fdf4706b3f48f1d95db1a4f5529b4d41'
EXPECTED = {'train.csv':'76a0a2d2f92b286371fe4d4044640910a04a803fdd2538e0f3f29a5c6f6b672e', 'test.csv':'521465c2428ed7f02f8d6db6ffdd4b5447c1c701962353eb2c40d548c3c85699'}

from IPython.display import display
REBUILD = True

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Environment: Jupyter Notebook")
print("Python:", sys.version.split()[0])
print("\nPackage versions:")

packages = ["numpy","pandas","scipy","scikit-learn","joblib","torch","sentence-transformers",
    "transformers","huggingface-hub","matplotlib","ipython","socksio",]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

In [ ]:
def save_json(path, obj):
    p = ROOT / path; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str)+'\n')

def load_json(path): return json.loads((ROOT/path).read_text())

def digest(value): return hashlib.sha256(value.encode()).hexdigest()

def normalize(text):
    text = html.unescape(unicodedata.normalize('NFKC', str(text))).casefold()
    return ' '.join(re.findall(r'[a-z0-9]+', text))

def validate_inputs():
    """Read bundled data locally; no dataset download or archive extraction."""
    for name, expected in EXPECTED.items():
        path = ROOT / name
        if name == 'train.csv':
            path = ROOT / 'train.csv.gz'
        if not path.is_file():
            raise FileNotFoundError(f'Keep all five project files together. Missing: {path.name}')
        h = hashlib.sha256()
        opener = gzip.open if path.suffix == '.gz' else open
        with opener(path, 'rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                h.update(chunk)
        if h.hexdigest() != expected:
            raise ValueError(f'Dataset checksum mismatch: {path.name}')
    return ['World', 'Sports', 'Business', 'Sci/Tech']

## Data audit and splits
Official test identities are held out. Exact and conservative near-duplicate groups are connected; linked training records and conflicting-label groups are removed. One consistent representative remains per group. Near-duplicate rule: at least six unique tokens, deletion signatures, Jaccard ≥ 0.8 and character similarity ≥ 0.92. It can miss paraphrases. Training/development split: stratified 90/10, seed 42. Description is retained for provenance but never passed to either model.

In [ ]:
def prepare_data():
    """Preserve official test; remove linked train rows and duplicate representatives."""
    for directory in ['outputs/results', 'outputs/artifacts', 'outputs/processed']:
        (ROOT / directory).mkdir(parents=True, exist_ok=True)
    validate_inputs(); records=[]; removed=[]; before={}
    for split in ['train','test']:
        n=0
        opener = gzip.open if split == 'train' else open
        path = ROOT / ('train.csv.gz' if split == 'train' else 'test.csv')
        with opener(path, 'rt', newline='', encoding='utf-8') as f:
            for i,row in enumerate(csv.reader(f)):
                sid=f'{split}:{i+1:06d}'
                if len(row)!=3 or row[0] not in ['1','2','3','4']:
                    removed.append({'id':sid,'reason':'malformed record'}); continue
                if row[0] not in ['3','4']: continue
                n+=1
                text=row[1]  # Original title retained. Never split a combined text field.
                if not normalize(text): removed.append({'id':sid,'reason':'empty normalized title'}); continue
                records.append({'id':sid,'official_split':split,'label':int(row[0])-3,'title':text,'description':row[2],'fingerprint':digest(normalize(text)),'normalized':normalize(text)})
        before[split]=n
    df=pd.DataFrame(records); parent=list(range(len(df)))
    def find(i):
        while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(i,j):
        a,b=find(i),find(j)
        if a!=b: parent[max(a,b)]=min(a,b)
    exact={}; buckets=defaultdict(list); near=[]
    # A reproducible conservative near-duplicate rule. Identical token sets or one
    # token substitution/deletion with >= 6 unique tokens, Jaccard >= .8 and
    # character SequenceMatcher similarity >= .92. It cannot catch all paraphrases.
    from difflib import SequenceMatcher
    token_sets=[set(x.split()) for x in df.normalized]
    for i,row in df.iterrows():
        fp=row.fingerprint
        if fp in exact: union(i,exact[fp])
        else: exact[fp]=i
        ts=sorted(token_sets[i])
        if len(ts)<6: continue
        signatures={tuple(ts)} | {tuple(ts[:j]+ts[j+1:]) for j in range(len(ts))}
        candidates=set()
        for sig in signatures: candidates.update(buckets[sig])
        for j in sorted(candidates):
            a,b=token_sets[i],token_sets[j]
            jac=len(a&b)/len(a|b)
            if jac>=.8 and SequenceMatcher(None,row.normalized,df.iloc[j].normalized,autojunk=False).ratio()>=.92:
                union(i,j); near.append({'id_a':df.iloc[j].id,'id_b':row.id,'jaccard':jac})
        # Store only exact-text representatives to limit redundant comparisons.
        if exact[fp]==i:
            for sig in signatures: buckets[sig].append(i)
    df['group']=[f'g{find(i):06d}' for i in range(len(df))]
    keep=[]
    for group,g in df.groupby('group',sort=True):
        test=g[g.official_split=='test']
        if g.label.nunique()>1:
            for sid in g.id: removed.append({'id':sid,'reason':'conflicting-label duplicate group','group':group})
            continue
        chosen=(test if len(test) else g).iloc[0]
        keep.append(chosen.name)
        for _,r in g.iterrows():
            if r.name==chosen.name: continue
            why='linked to official test' if len(test) and r.official_split=='train' else 'repeated duplicate group'
            removed.append({'id':r.id,'reason':why,'group':group,'retained_id':chosen.id})
    clean=df.loc[keep].copy(); tr=clean[clean.official_split=='train']
    train,dev=train_test_split(tr,test_size=.1,random_state=SEED,stratify=tr.label)
    test=clean[clean.official_split=='test']
    splits={'train':train.sort_values('id'),'dev':dev.sort_values('id'),'test':test.sort_values('id')}
    for a,b in [('train','dev'),('train','test'),('dev','test')]:
        assert not set(splits[a].id)&set(splits[b].id)
        assert not set(splits[a].group)&set(splits[b].group)
        assert not set(splits[a].fingerprint)&set(splits[b].fingerprint)
    out=ROOT/'outputs/processed';out.mkdir(parents=True,exist_ok=True)
    for name,s in splits.items(): s.to_csv(out/f'{name}.csv',index=False)
    pd.DataFrame(removed).to_csv(ROOT/'outputs/results/removals.csv',index=False)
    pd.DataFrame(near,columns=['id_a','id_b','jaccard']).to_csv(ROOT/'outputs/results/near_duplicate_pairs.csv',index=False)
    counts={k:{'total':len(v),'classes':{LABELS[int(y)]:int(n) for y,n in v.label.value_counts().items()}} for k,v in splits.items()}
    save_json('outputs/results/data_audit.json',{'before_selected':before,'after':counts,'removal_count':len(removed),'removal_reasons':pd.Series([r['reason'] for r in removed]).value_counts().to_dict(),'near_duplicate_links':len(near),'checks':'ID, fingerprint and connected duplicate groups disjoint across all splits','test_population':'One representative per consistent duplicate group; conflicting groups excluded; official test records never moved to training.'})
    return splits

def splits():
    return {n: pd.read_csv(ROOT/f'outputs/processed/{n}.csv', keep_default_na=False)
            for n in ['train', 'dev', 'test']}

In [ ]:
if REBUILD:
    data = prepare_data()
else:
    validate_inputs()
    data = splits()
for a, b in [('train','dev'), ('train','test'), ('dev','test')]:
    for field in ['id','group','fingerprint']:
        assert not set(data[a][field]) & set(data[b][field])
display(pd.DataFrame({name: d.label.value_counts() for name,d in data.items()}).rename(index=dict(enumerate(LABELS))))
print(json.dumps(load_json('outputs/results/data_audit.json'), indent=2))

## Two separate lexicons
The supplied CSVs are the frozen v1.0 term lists derived from training candidate statistics and AI-assisted semantic review of training candidates and a seeded development sample, without model predictions. Loading them replays documented choices; it does not constitute new student review. Science & Technology includes computing, technology and natural science.

Whole-token/phrase matching uses NFKC normalization, HTML unescaping, lowercase and alphanumeric tokens. A headline is **overlap** if it matches at least one term from each list. This is an analysis rule, not an input dictionary used to train the classifiers. Polysemy (e.g. shares, rates, virus) creates false positives. Do not revise these lists based on observed test results.

In [ ]:
def load_lexicons():
    result = {}
    for label, filename in zip(LABELS, ['business_lexicon.csv','science_tech_lexicon.csv']):
        table = pd.read_csv(ROOT/filename, keep_default_na=False)
        if 'term' not in table or table.empty:
            raise ValueError(f'{filename} must contain a non-empty term column.')
        terms = [normalize(t) for t in table.term]
        if any(not t for t in terms) or len(set(terms)) != len(terms):
            raise ValueError(f'{filename} contains blank or duplicate terms.')
        result[label] = sorted(terms)
    return result

In [ ]:
def lexicon_candidates():
    tr=splits()['train']
    cv=CountVectorizer(ngram_range=(1,2),min_df=15,stop_words='english',binary=True,token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')
    x=cv.fit_transform(tr.title); terms=cv.get_feature_names_out(); counts=np.vstack([np.asarray(x[tr.label.to_numpy()==y].sum(axis=0)).ravel() for y in [0,1]])
    ns=np.array([(tr.label==y).sum() for y in [0,1]])
    odds=np.log((counts+.5)/(ns[:,None]-counts+.5)); rows=[]
    for y in [0,1]:
        score=odds[y]-odds[1-y]; idx=np.argsort(-score)
        chosen=list(idx[:120]); chosen += [j for j in np.argsort(-counts[y]) if score[j]>.35 and j not in chosen][:80]
        for j in chosen: rows.append({'label':LABELS[y],'term':terms[j],'document_frequency':int(counts[y,j]),'other_frequency':int(counts[1-y,j]),'log_odds_difference':float(score[j])})
    pd.DataFrame(rows).to_csv(ROOT/'outputs/results/lexicon_candidates.csv',index=False)
    return pd.DataFrame(rows)

def match_terms(text,lexicon):
    s=' '+normalize(text)+' '
    return {label:[t for t in terms if ' '+normalize(t)+' ' in s] for label,terms in lexicon.items()}

def annotate(df,lexicon):
    d=df.copy(); m=[match_terms(t,lexicon) for t in d.title]
    d['business_terms']=[json.dumps(x[LABELS[0]]) for x in m]
    d['science_terms']=[json.dumps(x[LABELS[1]]) for x in m]
    d['overlap']=[bool(x[LABELS[0]] and x[LABELS[1]]) for x in m]
    return d

In [ ]:
def build_review():
    c=pd.read_csv(ROOT/'outputs/results/lexicon_candidates.csv'); requested=load_lexicons();lex={}
    review=[]
    for label,terms in requested.items():
        available=set(c[c.label==label].term)
        missing=set(terms)-available
        assert not missing, missing
        lex[label]=sorted(terms)
    for _,r in c.iterrows():
        selected=r.term in lex[r.label]
        reason='Retained: financial/economic topic term' if r.label==LABELS[0] else 'Retained: technology/computing/natural-science topic term'
        if not selected:reason='Excluded: generic context, publisher/source, company/person entity, ambiguous fragment or overly specific/duplicate phrase'
        review.append({**r.to_dict(),'decision':'retain' if selected else 'exclude','reason':reason})
    pd.DataFrame(review).to_csv(ROOT/'outputs/results/lexicon_term_review.csv',index=False)
    save_json('outputs/artifacts/lexicons.json',lex)
    dev=annotate(splits()['dev'],lex);sample=dev[dev.overlap].sample(n=min(20,int(dev.overlap.sum())),random_state=SEED).sort_values('id')
    sample[['id','title','business_terms','science_terms']].to_csv(ROOT/'outputs/results/development_overlap_review.csv',index=False)
    save_json('outputs/artifacts/lexicon_review.json',{'version':'1.0','source_split':'train only','source_manifest_sha256':hashlib.sha256((ROOT/'outputs/processed/train.csv').read_bytes()).hexdigest(),'normalization':'NFKC, HTML unescape, casefold, alphanumeric tokens, whitespace collapse','ranking':'class document-presence log odds with 0.5 smoothing; top 120 log-odds plus 80 highest-frequency remaining terms with positive log odds >0.35; min_df=15','stopwords':'sklearn English candidate extraction only; classifiers keep stopwords','decision_author':'AI-assisted researcher review; students must review before claiming authorship','dev_overlap_n':int(dev.overlap.sum()),'dev_class_counts':dev[dev.overlap].label.value_counts().to_dict(),'dev_sample':'outputs/results/development_overlap_review.csv','dev_review_status':'sample prepared for semantic review before freeze','test_consulted':False,'predictions_used':False,'limitations':['proxy for mixed vocabulary, not semantic ambiguity','entity names excluded so technology companies alone do not count','polysemy: virus, security, bank and shares can have multiple senses','class frequency selection and 2004-era dataset limit coverage','natural science terms included when supported by training']})
    print(sample[['id','title','business_terms','science_terms']].to_string(index=False))
    print('Overlap development',int(dev.overlap.sum()),'of',len(dev))


def finish_review():
    p=ROOT/'outputs/results/development_overlap_review.csv';d=pd.read_csv(p)
    comments={
      'train:019091':('polysemy false positive','Rates refers to divorce frequency, not financial interest rates. Retain rates for finance and disclose this lexical false positive.'),
      'train:078941':('polysemy false positive','Shares means tells others about dreams, not financial shares. Retain shares globally because it is a central financial term; disclose proxy noise.'),
      'train:087762':('metaphorical business wording','Bank on is figurative. Business topic remains plausible because sneaker makers are firms; proxy alone cannot settle semantic ambiguity.'),
      'train:009813':('entity ambiguity','China Mobile is a company name. Mobile matches lexical technology vocabulary without proving a product-focused story.')
    }
    d['review_judgement']=[comments.get(x,('plausible mixed vocabulary','Business transaction/price/performance language co-occurs with a technology or science term.'))[0] for x in d.id]
    d['review_note']=[comments.get(x,('plausible mixed vocabulary','Business transaction/price/performance language co-occurs with a technology or science term.'))[1] for x in d.id]
    d.to_csv(p,index=False)
    r=load_json('outputs/artifacts/lexicon_review.json');r['dev_review_status']='Reviewed 20 seeded development headlines semantically, without predictions; retained central financial terms despite documented polysemy.';r['revision_decisions']=['Initial seeded sample included train:011065 NASA Experts Report Drought Forecast Breakthrough. Removed forecast because it is generic. Re-reviewed seeded sample after this single change; no test or predictions consulted.'];r['decision_rule']='Use frozen v1.0 lexical proxy with explicit polysemy limitations; do not redesign using test.'
    save_json('outputs/artifacts/lexicon_review.json',r)

In [ ]:
if REBUILD:
    lexicon_candidates()
    build_review()
    finish_review()
lexicons = load_json('outputs/artifacts/lexicons.json')
assert lexicons == load_lexicons(), 'Lexicon inputs changed: rebuild before evaluation.'
print({label: len(terms) for label, terms in lexicons.items()})
display(pd.read_csv(ROOT/'outputs/results/development_overlap_review.csv').head(10))